# Geomap: Kongruenz nach Zeitphase

Wie stark stimmt die Bevölkerung in den einzelnen Kantonen mit Bundesrat, Bundesversammlung und Parteien überein – und verändert sich das über die historischen Phasen?

**Was passiert hier?**
- Heatmap-Datensatz nach Zeitphase laden (aus `3c_2_heatmap.ipynb`)
- Für jeden Akteur und jede Phase die kantonalen Zustimmungswerte ins Kartenformat bringen
- Interaktive Schweizer Karte mit Phasen- und Akteursauswahl erstellen
- Karte als HTML-Datei für den Blog exportieren

**Datengrundlage**
- `data/processed/df_heatmap_by_phase.csv` (aus `3c_2_heatmap.ipynb`)

**Vorher ausführen**
- `1_data_wrangling.ipynb` → `2_berechnung.ipynb` → `3c_2_heatmap.ipynb`

**Danach**
- Blog-Plot: `d7_geomap_zeitphasen_bv.html`
- Kein Pflicht-Folgenotebook; parallel zu `3a_`, `3b_`, `3d_`

# Setup
Bibliotheken und Hilfsfunktionen aus `visualisierungen.py` laden.

In [ ]:
# autoreload: Änderungen in visualisierungen.py werden sofort übernommen ohne Kernel-Neustart
%load_ext autoreload
%autoreload 2

import importlib
import re
from pathlib import Path

import pandas as pd
import visualisierungen

importlib.reload(visualisierungen)

# Nur die drei Funktionen laden die wir brauchen
from visualisierungen import (
    phase_kantons_row_to_map_df,       # wandelt eine CSV-Zeile in ein kartenlesbares Format um
    schweiz_karte_interaktiv_phasen,   # erstellt die interaktive Plotly-Karte
    write_plotly_html_responsive,      # exportiert die Karte als responsive HTML für den Blog
)

# Daten laden und vorbereiten
Heatmap-Datensatz einlesen und für jeden Akteur und jede Phase ins Kartenformat bringen.

In [ ]:
# Standardakteur beim ersten Laden der Karte: Bundesversammlung
PARTEI_BV = "bv-pos_label"

# Lesbarer Titel pro Phase. Erscheint als Beschriftung in der Grafik und auf den Phasen-Knöpfen
PHASE_TITLES = {
    "phase1_fruehphase": "Frühphase\n(1848-1899)",
    "phase2_volatile": "Volatile Phase\n(1900-1949)",
    "phase3_konsens": "Konsensphase\n(1950-1975)",
    "phase4_aufspaltung": "Aufspaltung\n(1976-2009)",
    "phase5_2010_heute": "2010er–heute\n(2010-heute)",
}


def _phase_order(slug: str) -> int:
    # Phasen anhand der eingebetteten Zahl sortieren statt alphabetisch
    # ohne diese Funktion würde z.B. phase5 vor phase1 landen
    m = re.search(r"phase(\d+)", str(slug))
    return int(m.group(1)) if m else 99


# CSV suchen. Funktioniert egal ob das Notebook von notebooks/ oder vom Projektroot gestartet wird
for base in (Path(".."), Path(".")):
    csv_path = base / "data" / "processed" / "df_heatmap_by_phase.csv"
    if csv_path.is_file():
        break
else:
    raise FileNotFoundError("df_heatmap_by_phase.csv nicht gefunden.")

df_phase = pd.read_csv(csv_path)
# Überflüssige Index-Spalte entfernen – entsteht beim CSV-Export ohne index=False
if "Unnamed: 0" in df_phase.columns:
    df_phase = df_phase.drop(columns=["Unnamed: 0"])

# Nur Zeilen der Bundesversammlung herausfiltern = als Standardakteur beim Seitenaufruf
bv = df_phase[df_phase["partei"] == PARTEI_BV]
# Phasen in chronologischer Reihenfolge (phase1, phase2, ...) sortieren
phase_slugs = sorted(bv["phase"].dropna().unique(), key=_phase_order)
PHASES = [(slug, PHASE_TITLES.get(slug, slug)) for slug in phase_slugs]

# Für jede Phase eine Zeile aus dem DataFrame holen und in ein kartenlesbares Format umwandeln
phasen_map = []
for slug, titel in PHASES:
    row = bv.loc[bv["phase"] == slug].iloc[0]
    phasen_map.append((titel, phase_kantons_row_to_map_df(row)))

# Akteure in gewünschter Dropdown-Reihenfolge
AKTEUR_ORDER = [
    "br-pos_label",
    "bv-pos_label",
    "p-gps_label",
    "p-sps_label",
    "p-mitte_label",
    "p-fdp_label",
    "p-svp_label",
]
AKTEUR_LABELS = {
    "br-pos_label": "Bundesrat",
    "bv-pos_label": "Bundesversammlung",
    "p-gps_label": "Grüne",
    "p-sps_label": "SP",
    "p-mitte_label": "Mitte",
    "p-fdp_label": "FDP",
    "p-svp_label": "SVP",
}
DEFAULT_AKTEUR = AKTEUR_LABELS[PARTEI_BV]

# Pro Akteur eine Liste von (Phasentitel, Karten-DataFrame) zusammenbauen
# → die Karte kann damit per Dropdown zwischen Akteuren wechseln ohne neu zu laden
akteur_phasen = {}
for partei_slug in AKTEUR_ORDER:
    sub = df_phase[df_phase["partei"] == partei_slug]
    if sub.empty:
        continue
    akteur_phasen[AKTEUR_LABELS.get(partei_slug, partei_slug)] = [
        (titel, phase_kantons_row_to_map_df(sub.loc[sub["phase"] == slug].iloc[0]))
        for slug, titel in PHASES
        if not sub.loc[sub["phase"] == slug].empty
    ]

# Kontrollausgabe: wie viele Akteure wurden geladen und welche?
len(akteur_phasen), list(akteur_phasen)

# Final Plot D7: Interaktive Grafik (Blog)
Schweizer Karte mit Phasen- und Akteursauswahl bauen und als `d7_geomap_zeitphasen_bv.html` für die Website exportieren.

In [ ]:
# Interaktive Karte erstellen – Nutzer kann Phase und Akteur per Klick wechseln
fig_bv = schweiz_karte_interaktiv_phasen(
    akteur_phasen=akteur_phasen,
    default_akteur=DEFAULT_AKTEUR,  # Bundesversammlung als Startzustand
    height=360,
)
fig_bv.show()

# Als responsive HTML exportieren damit die Karte im Blog auf allen Bildschirmgrössen passt
write_plotly_html_responsive(
    fig_bv,
    "../Blog/blog_plots/d7_geomap_zeitphasen_bv.html",
    height=400,
    # Phasentitel für die Auswahl-Knöpfe unter der Karte
    phase_bar_labels=[titel for titel, _ in phasen_map],
    phase_bar_layout="generic",
)